In [1]:
import warnings
warnings.filterwarnings('ignore')

import os, time, requests
from io import StringIO
from datetime import datetime
from tqdm.notebook import tqdm
from joblib import Parallel, delayed

import numpy as np
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import lightgbm as lgb

# ── Point this at wherever your main notebook saved its outputs ────
MODEL_DIR   = 'nse_lgbm_output'   # change if saved elsewhere

TODAY       = datetime.today().strftime('%Y-%m-%d')
OHLCV_START = '2019-06-01'        # need history for 26w indicators
TARGET_PCT  = 0.12
SL_PCT      = 0.03
HOLD_DAYS   = 3
TOP_N       = 3
MKTCAP_MIN  = 500

FEATURE_COLS = [
    'atr_squeeze', 'bb_squeeze', 'near_high', 'vol_momentum',
    'momentum_2w', 'ema_dip', 'rsi_oversold', 'trend_26w',
    'stoch_oversold', 'obv_accel', 'macd_strength',
]

print(f'✅ Config ready | Today: {TODAY}')
print(f'📂 Loading model from: {MODEL_DIR}/')

✅ Config ready | Today: 2026-03-23
📂 Loading model from: nse_lgbm_output/


In [2]:
# ── Load LightGBM model ───────────────────────────────────────────
model_path = os.path.join(MODEL_DIR, 'lgbm_model.txt')
assert os.path.exists(model_path), (
    f"Model not found at {model_path}. "
    f"Run the main training notebook first."
)
lgbm_model = lgb.Booster(model_file=model_path)
print(f'✅ Model loaded — {lgbm_model.num_trees()} trees')

# ── Load learned weights (W_approx) ──────────────────────────────
weights_path = os.path.join(MODEL_DIR, 'learned_weights.csv')
weights_df   = pd.read_csv(weights_path)
# Reconstruct W_approx in FEATURE_COLS order
w_map   = weights_df.set_index('Feature')['Proxy Weight'].to_dict()
W_approx = np.array([w_map[f] for f in FEATURE_COLS], dtype=np.float32)
print(f'✅ Weights loaded')

# ── Load feature importance for display ──────────────────────────
imp_path     = os.path.join(MODEL_DIR, 'feature_importance.csv')
importance_df = pd.read_csv(imp_path)
imp_norm      = importance_df.set_index('Indicator')['Gain_norm'] * 100

# ── Print weight table ────────────────────────────────────────────
print()
print('  Feature Weights (from training notebook):')
print(f'  {"Feature":<22} {"Gain%":>7}  {"Dir":>4}  {"Proxy W%":>9}')
print(f'  {"-"*48}')
for _, r in weights_df.sort_values('Gain %', ascending=False).iterrows():
    bar = '█' * max(1, int(r['Gain %'] / 3))
    print(f'  {r["Feature"]:<22} {r["Gain %"]:>6.1f}%  '
          f'{r["Direction"]:>4}  {r["Proxy W %"]:>8.2f}%  {bar}')

✅ Model loaded — 185 trees
✅ Weights loaded

  Feature Weights (from training notebook):
  Feature                  Gain%   Dir   Proxy W%
  ------------------------------------------------
  near_high                21.8%     +     21.81%  ███████
  momentum_2w              21.1%     +     21.07%  ███████
  atr_squeeze              12.4%     +     12.36%  ████
  trend_26w                 9.6%     +      9.61%  ███
  vol_momentum              7.8%     +      7.84%  ██
  obv_accel                 7.5%     +      7.47%  ██
  macd_strength             5.3%     +      5.34%  █
  rsi_oversold              4.5%     −      4.46%  █
  stoch_oversold            4.2%     −      4.17%  █
  bb_squeeze                3.3%     −      3.34%  █
  ema_dip                   2.5%     −      2.53%  █


In [3]:
def fetch_nse_symbols():
    url     = 'https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv'
    headers = {'User-Agent': 'Mozilla/5.0',
               'Referer'  : 'https://www.nseindia.com'}
    try:
        s = requests.Session()
        s.get('https://www.nseindia.com', headers=headers, timeout=10)
        r = s.get(url, headers=headers, timeout=30)
        r.raise_for_status()
        df  = pd.read_csv(StringIO(r.text))
        col = [c for c in df.columns if 'SYMBOL' in c.upper()][0]
        syms = df[col].dropna().str.strip().tolist()
        print(f'✅ NSE: {len(syms)} symbols')
        return syms
    except Exception as e:
        print(f'⚠️  NSE fetch failed ({e})')
        return []

nse_symbols = fetch_nse_symbols()

✅ NSE: 2272 symbols


In [12]:
# Uses the cached mcap file from the main notebook — no re-fetching needed
MCAP_CACHE = os.path.join(MODEL_DIR, 'mcap_cache.csv')

if os.path.exists(MCAP_CACHE):
    mcap_df   = pd.read_csv(MCAP_CACHE, index_col=0)
    mcap_dict = mcap_df['mcap_cr'].to_dict()
    print(f'✅ Loaded {len(mcap_dict)} cached market caps from {MCAP_CACHE}')
else:
    print(f'⚠️  mcap_cache.csv not found in {MODEL_DIR}/')
    print('   Run the main notebook once to generate it, then re-run this cell.')
    mcap_dict = {}

# Filter symbols to those >= 500 Cr
qualified_syms = [
    s for s in nse_symbols
    if mcap_dict.get(s) is not None and mcap_dict[s] >= MKTCAP_MIN
]
print(f'   Universe after mkt cap filter: {len(qualified_syms)} stocks')

✅ Loaded 2217 cached market caps from nse_lgbm_output\mcap_cache.csv
   Universe after mkt cap filter: 630 stocks


In [13]:
def download_ohlcv(symbols, start=OHLCV_START, end=TODAY,
                   suffix='.NS', batch=50):
    result, failed = {}, []
    tickers = [s + suffix for s in symbols]
    for i in tqdm(range(0, len(tickers), batch), desc='OHLCV'):
        grp = tickers[i:i+batch]
        try:
            raw = yf.download(
                grp, start=start, end=end,
                group_by='ticker', auto_adjust=True,
                progress=False, threads=True
            )
            for t in grp:
                sym = t.replace(suffix, '')
                try:
                    df = raw[t].copy() if len(grp) > 1 else raw.copy()
                    df.dropna(subset=['Close'], inplace=True)
                    if len(df) >= 52:
                        df.columns = [c.lower() for c in df.columns]
                        result[sym] = df
                    else:
                        failed.append(sym)
                except Exception:
                    failed.append(sym)
        except Exception as e:
            print(f'Batch error: {e}')
        time.sleep(0.3)
    print(f'✅ Downloaded: {len(result)} | Skipped: {len(failed)}')
    return result

print(f'⬇️  Downloading OHLCV ({OHLCV_START} → {TODAY})...')
ohlcv_data = download_ohlcv(qualified_syms)
print(f'   Universe: {len(ohlcv_data)} stocks')

⬇️  Downloading OHLCV (2019-06-01 → 2026-03-23)...


OHLCV:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Downloaded: 630 | Skipped: 0
   Universe: 630 stocks


In [36]:
print('📡 Downloading Nifty 50 for regime filter...')
nifty_raw  = yf.download('^NSEI', start=OHLCV_START, end=TODAY,
                          auto_adjust=True, progress=False)
nifty_close = nifty_raw['Close'].squeeze().rename('nifty').dropna()
nifty_close.index = pd.to_datetime(nifty_close.index)

# ── Regime indicators ──────────────────────────────────────────────
nifty_df            = pd.DataFrame({'close': nifty_close})
nifty_df['ema20']   = nifty_df['close'].ewm(span=20,  adjust=False).mean()
nifty_df['ema50']  = nifty_df['close'].ewm(span=50, adjust=False).mean()

# ── Classify each trading day ──────────────────────────────────────
bull = (nifty_df['close'] > nifty_df['ema20'])  & (nifty_df['ema20'] >= nifty_df['ema50'])
bear = (nifty_df['close'] < nifty_df['ema20'])  & (nifty_df['ema20'] < nifty_df['ema50'])
nifty_df['regime'] = np.select([bull, bear], ['BULLISH', 'BEARISH'], default='CAUTION')

# ── Boolean Series — forward-fillable lookup ───────────────────────
is_bullish = (nifty_df['regime'] == 'BULLISH') | (nifty_df['regime'] == 'CAUTION')

def regime_on(date_str: str) -> bool:
    """Return True if Nifty is in BULLISH regime on or before date_str."""
    try:
        val = is_bullish.loc[:pd.Timestamp(date_str)].iloc[-1]
        return bool(val)
    except Exception:
        return False

# ── Summary ────────────────────────────────────────────────────────
pct_b = (nifty_df['regime'] == 'BULLISH').mean() * 100
pct_c = (nifty_df['regime'] == 'CAUTION').mean() * 100
pct_r = (nifty_df['regime'] == 'BEARISH').mean() * 100


if current_regime == 'BEARISH':
    print(f'\n  ⛔  NO TRADES TODAY')
    print(f'     Strategy only operates in BULLISH regime.')
    print(f'     Come back when Nifty close > EMA50 > EMA200.')
    print(f'Nifty Close : {nifty_close[-1]:,.1f}')
    print(f'Nifty EMA20 : {nifty_df['ema20'][-1]:,.1f}')
    print(f'Nifty Close : {nifty_df['ema50'][-1]:,.1f}')

📡 Downloading Nifty 50 for regime filter...

  ⛔  NO TRADES TODAY
     Strategy only operates in BULLISH regime.
     Come back when Nifty close > EMA50 > EMA200.
Nifty Close : 23,114.5
Nifty EMA20 : 24,145.3
Nifty Close : 24,855.0


In [29]:
def compute_features(df):
    d = df.copy()
    d.columns = [c.lower() for c in d.columns]
    c, h, l, o, v = d['close'], d['high'], d['low'], d['open'], d['volume']

    atr_abs          = ta.atr(h, l, c, length=14)
    d['atr_pct']     = atr_abs / c
    atr_baseline     = d['atr_pct'].rolling(20).mean().replace(0, np.nan)
    d['atr_squeeze'] = np.clip(2*(1 - d['atr_pct']/atr_baseline), 0, 1).fillna(0)

    bb           = ta.bbands(c, length=20, std=2)
    bbb_col      = [col for col in bb.columns if col.startswith('BBB')][0]
    d['bb_width']= bb[bbb_col] / 100
    bb_baseline  = d['bb_width'].rolling(20).mean().replace(0, np.nan)
    d['bb_squeeze'] = np.clip(2*(1 - d['bb_width']/bb_baseline), 0, 1).fillna(0)

    d['resist_10w']    = h.rolling(10).max()
    d['pct_to_resist'] = (d['resist_10w'] - c) / c * 100
    d['near_high']     = np.clip(1 - d['pct_to_resist']/12, 0, 1).fillna(0)

    d['vol_ma20']     = v.rolling(20).mean()
    d['vol_ratio']    = v / d['vol_ma20']
    up_bar            = (c > c.shift(1)).astype(float)
    d['vol_momentum'] = (np.clip((d['vol_ratio']-0.8)/2.2, 0, 1)*(0.1+0.9*up_bar)).fillna(0)

    d['ret_2w']      = c.pct_change(2) * 100
    d['momentum_2w'] = np.clip(d['ret_2w']/5, 0, 1).fillna(0)

    d['ema_20']  = ta.ema(c, length=20)
    ema_dist     = (c - d['ema_20']) / d['ema_20'] * 100
    d['ema_dip'] = np.clip(-ema_dist/8, 0, 1).fillna(0)

    d['rsi_14']       = ta.rsi(c, length=14)
    d['rsi_oversold'] = np.clip((60 - d['rsi_14'])/40, 0, 1).fillna(0)

    d['ret_26w']   = c.pct_change(26) * 100
    d['trend_26w'] = np.clip(d['ret_26w']/40, 0, 1).fillna(0)

    stoch           = ta.stoch(h, l, c, k=14, d=3, smooth_k=3)
    stoch_d_col     = [col for col in stoch.columns if col.startswith('STOCHd')][0]
    d['stoch_d']    = stoch[stoch_d_col]
    d['stoch_oversold'] = np.clip((40 - d['stoch_d'])/35, 0, 1).fillna(0)

    d['obv']         = ta.obv(c, v)
    obv_chg          = d['obv'].diff(10)
    obv_z            = ((obv_chg - obv_chg.rolling(20).mean()) /
                        (obv_chg.rolling(20).std().replace(0,np.nan))).clip(-3,3).fillna(0)
    d['obv_accel']   = (obv_z + 3) / 6

    macd             = ta.macd(c, fast=12, slow=26, signal=9)
    macd_h_col       = [col for col in macd.columns if col.startswith('MACDh')][0]
    d['macd_hist']   = macd[macd_h_col]
    hist_z           = (d['macd_hist'] /
                        d['macd_hist'].rolling(20).std().replace(0,np.nan)).clip(-3,3).fillna(0)
    d['macd_strength'] = np.clip((hist_z + 1)/2, 0, 1)

    return d


print('⚙️  Computing features...')

def _worker(tk):
    try:
        fd = compute_features(ohlcv_data[tk])
        fd['symbol'] = tk
        return tk, fd
    except Exception:
        return tk, None

results = Parallel(n_jobs=-1, backend='threading')(
    delayed(_worker)(tk) for tk in tqdm(ohlcv_data, desc='Features')
)
feature_store = {tk: fd for tk, fd in results
                 if fd is not None and len(fd) > 100}
print(f'✅ Features ready for {len(feature_store)} symbols')

⚙️  Computing features...


Features:   0%|          | 0/630 [00:00<?, ?it/s]

✅ Features ready for 616 symbols


In [39]:
# ── Vectorized scorer ────────────────────────────────────────────
def score_universe(fstore, date_str, model, W):
    rows, tickers, closes = [], [], []
    for tk, df in fstore.items():
        sub = df[df.index <= date_str]
        if len(sub) < 30:
            continue
        feat_row = sub.iloc[-1][FEATURE_COLS].values.astype(np.float32)
        if np.any(np.isnan(feat_row)):
            continue
        rows.append(feat_row)
        tickers.append(tk)
        closes.append(float(sub.iloc[-1]['close']))
    if not rows:
        return pd.DataFrame()
    X_mat       = np.array(rows, dtype=np.float32)
    lgbm_scores = model.predict(X_mat)           # exact LGBM score
    proxy_scores= X_mat @ W                      # fast matrix multiply

    result = pd.DataFrame({
        'ticker'     : tickers,
        'lgbm_score' : lgbm_scores,
        'proxy_score': proxy_scores,
        'close'      : closes,
    })
    for i, col in enumerate(FEATURE_COLS):
        result[col] = X_mat[:, i]
    return (result.sort_values('lgbm_score', ascending=False)
                  .reset_index(drop=True)
                  .assign(rank=lambda df: df.index+1)
                  .set_index('rank'))


# ── Print recommendations ─────────────────────────────────────────
print(f'\n{"="*68}')
print(f'  🎯  NSE PAPER TRADE RECOMMENDATIONS  |  {TODAY}')
print(f'  📊  Target: +{TARGET_PCT*100:.0f}%  |  Stop-Loss: -{SL_PCT*100:.0f}%  |  Horizon: 2–3 days')
print(f'{"="*68}')

emoji = {'BULLISH':'🟢','CAUTION':'🟡','BEARISH':'🔴'}.get(current_regime,'⚪')
print(f'\n  {emoji}  Nifty Regime: {current_regime}  '
      f'(Close {nifty_close[-1]:,.0f} | EMA20 {nifty_df['ema20'][-1]:,.0f} | EMA200 {nifty_df['ema50'][-1]:,.0f})')

if current_regime != 'BULLISH':
    print(f'\n  ⛔  NO TRADES TODAY — market is {current_regime}.')
    print(f'     Check back when Nifty is BULLISH (Close > EMA20 > EMA50).')
else:
    scores = score_universe(feature_store, TODAY, lgbm_model, W_approx)

    if scores.empty:
        print('⚠️  Could not generate scores — check data')
    else:
        top5 = scores.head(5).copy()
        top5['target'] = (top5['close'] * (1 + TARGET_PCT)).round(2)
        top5['sl']     = (top5['close'] * (1 - SL_PCT)).round(2)
        top5['rr']     = f'{int(TARGET_PCT/SL_PCT)}:1'

        # ── Clean recommendation table ────────────────────────────
        print(f'\n  {"#":<3} {"Ticker":<14} {"CMP (₹)":>10} '
              f'{"Target (₹)":>12} {"SL (₹)":>10} {"Score":>10}')
        print(f'  {"-"*62}')
        for i, (_, r) in enumerate(top5.iterrows(), 1):
            print(f'  #{i:<2} {r["ticker"]:<14} ₹{r["close"]:>9,.2f} '
                  f'₹{r["target"]:>11,.2f} ₹{r["sl"]:>9,.2f} '
                  f'{r["lgbm_score"]:>10.5f}')

        print(f'\n  R:R = {int(TARGET_PCT/SL_PCT)}:1  |  '
              f'Hold up to {HOLD_DAYS} days  |  Use CNC orders')
        print(f'  Place GTT OCO: SL at -3% and TP at +12% immediately after entry fills.')

        # ── Indicator breakdown ───────────────────────────────────
        print(f'\n  {"─"*65}')
        print(f'  Indicator breakdown (what drove the score):')
        print(f'  {"─"*65}')
        for i, (_, r) in enumerate(top5.iterrows(), 1):
            print(f'\n  #{i} {r["ticker"]}  |  LGBM score: {r["lgbm_score"]:.5f}')
            for feat in FEATURE_COLS:
                val  = r[feat]
                gain = imp_norm.get(feat, 0)
                bar  = ('█' * min(int(abs(val)*5+1), 8)) if not np.isnan(val) else '?'
                arr  = '▲' if val > 0 else ('▼' if val < 0 else '─')
                print(f'     {feat:<22} {val:>7.4f}  {arr} {bar:<8}  '
                      f'(model weight: {gain:.1f}%)')

        print(f'\n{"="*68}')
        print('  ⚠️  Paper trade only. Not SEBI investment advice.')
        print(f'{"="*68}')

        # Store for journal cell below
        today_recs = top5.copy()


  🎯  NSE PAPER TRADE RECOMMENDATIONS  |  2026-03-23
  📊  Target: +12%  |  Stop-Loss: -3%  |  Horizon: 2–3 days

  🔴  Nifty Regime: BEARISH  (Close 23,114 | EMA20 24,145 | EMA200 24,855)

  ⛔  NO TRADES TODAY — market is BEARISH.
     Check back when Nifty is BULLISH (Close > EMA20 > EMA50).
